### Why Deep Learning Needs GPUs
Recall that neural networks mainly perform:
- Matrix Multiplication
- Tensor Operations

Example:
```
Input Tensor
      ×
Weight Matrix
      ↓
Output Tensor
```

Modern models perform:
- Billions of operations
- Trillions of operations

during training.

### CPU vs GPU
#### CPU (Central Processing Unit)
Designed for:
```
General Purpose Tasks
```

Examples:
- Operating System
- Web Browser
- Databases
- Applications
- Business Logic

A CPU has relatively few powerful cores.

Diagram:
```
CPU

Core 1
Core 2
Core 3
Core 4
```

Good for:
```
Complex Sequential Tasks
```

Examples:
- Running programs
- Decision-making logic
- Operating systems

#### GPU (Graphics Processing Unit)
Designed for:
```
Massive Parallel Computation
```

Diagram:
```
GPU

■■■■■■■■■■■■■■■■■■
■■■■■■■■■■■■■■■■■■
■■■■■■■■■■■■■■■■■■
```
Thousands of smaller cores work simultaneously.

Good for:
- Matrix Operations
- Tensor Operations
- Parallel Computation


### Why GPU Wins
Suppose:
```
1 Million Multiplications
```

CPU:
```
Few Powerful Workers
```

GPU:
```
Thousands Of Workers
```

Diagram:
```
CPU

Task
 ↓
Task
 ↓
Task
 ↓
Task
```
```
GPU

Task Task Task Task
Task Task Task Task
Task Task Task Task
```

The GPU executes many operations simultaneously.

### Deep Learning Perspective
Neural network:
```
Input
 ↓
Layer 1
 ↓
Layer 2
 ↓
Layer 3
 ↓
Output
```

Each layer performs:
```
Matrix Multiplication
      +
Bias Addition
      +
Activation
```

GPUs accelerate these operations dramatically.


### Example: Matrix Multiplication
Suppose:
```
(1024 × 2048)
        ×
(2048 × 4096)
```

This involves billions of arithmetic operations.

CPU:
```
Slow
```

GPU:
```
Highly Parallel
```

Huge speed improvement.

### CUDA
**Definition**

CUDA is NVIDIA's GPU computing platform.

It allows PyTorch to use NVIDIA GPUs.

Diagram:
```
PyTorch
    ↓
CUDA
    ↓
NVIDIA GPU
```

Without CUDA:

```
PyTorch
    ↓
CPU
```

### Check GPU Availability
```python
import torch

print(
    torch.cuda.is_available()
)
```

Output:
```python
True
```

or
```python
False
```

##### Additional GPU Information
```python
print(
    torch.cuda.device_count()
)
```

Example Output:
```python
1
```

##### GPU Name
```python
print(
    torch.cuda.get_device_name(0)
)
```

Example:
```python
Tesla T4
```

or
```python
NVIDIA A100
```

### Device Concept
PyTorch uses:
```
device
```

to determine where tensors and models are stored.

Possible values:
```python
"cpu"
```

```python
"cuda"
```

### Selecting Device
Standard approach:
```python
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)
```

Benefits:
- Works on GPU machines
- Works on CPU machines
- Production-safe

### Move Tensor To GPU
Create tensor:
```python
x = torch.tensor(
    [1, 2, 3]
)
```

Stored on:

```text
CPU
```

Move:
```python
x = x.to(device)
```

Diagram:
```
CPU Tensor
      ↓
 x.to(device)
      ↓
GPU Tensor
```

### Move Model To GPU
Very important.
```python
model = model.to(device)
```

Diagram:
```
Model
  ↓
GPU Memory
```

All model parameters now reside on the GPU.

### Important Rule
Both model and data must be on the same device.

Correct:

```python
model = model.to(device)

X_batch = X_batch.to(device)
```

Wrong:
```
Model → GPU

Data → CPU
```

Produces:
```
RuntimeError:
Expected all tensors to be on the same device
```

### Typical Training Loop With GPU
```python
for X_batch, y_batch in loader:

    X_batch = X_batch.to(device)

    y_batch = y_batch.to(device)

    optimizer.zero_grad()

    predictions = model(
        X_batch
    )

    loss = criterion(
        predictions,
        y_batch
    )

    loss.backward()

    optimizer.step()
```

Only the `.to(device)` lines are added compared to CPU training.

### Checking Model Device
```python
print(
    next(
        model.parameters()
    ).device
)
```

Output:

```python
cuda:0
```

if the model is on GPU.

### GPU Memory
GPUs have limited memory.

Examples:
| GPU | Memory |
|------|---------|
| T4 | 16 GB |
| L4 | 24 GB |
| V100 | 16–32 GB |
| A100 | 40–80 GB |
| H100 | 80+ GB |

Large models consume enormous memory.

### What Uses GPU Memory?
Main consumers:
```text
Model Parameters
Activations
Gradients
Optimizer States
Mini-Batches
```

Diagram:
```
GPU Memory

├── Model
├── Activations
├── Gradients
├── Optimizer
└── Batch Data
```

### Out Of Memory (OOM)
Common error:
```
CUDA Out Of Memory
```

Example:
```
RuntimeError:
CUDA out of memory
```

Caused by:
- Batch size too large
- Model too large
- Sequence length too large
- Multiple models loaded

### Typical Fixes
#### Fix 1: Reduce Batch Size
Before:
```python
batch_size = 512
```

After:
```python
batch_size = 128
```

Most common solution.

#### Fix 2: Reduce Model Size
Before:
```
Hidden Layer = 4096
```

After:
```
Hidden Layer = 1024
```

#### Fix 3: Use Gradient Accumulation
Advanced technique.

Idea:
```
Small Batches
      ↓
Accumulate Gradients
      ↓
Large Effective Batch
```

#### Fix 4: Clear GPU Cache
```python
torch.cuda.empty_cache()
```

Useful during experimentation.

### CPU vs GPU Speed Example
Training a simple network:
| Hardware | Time |
|-----------|---------|
| CPU | 2 Hours |
| T4 GPU | 10 Minutes |
| A100 GPU | 2–3 Minutes |

Actual numbers vary but the difference is often dramatic.

### Multi-GPU Training
Large models may use:
```
GPU 1
GPU 2
GPU 3
GPU 4
```

Diagram:
```
Dataset
   ↓
GPU1 GPU2 GPU3 GPU4
   ↓
Combined Gradients
```

Used for:
- GPT
- BERT
- LLaMA
- Large Transformers


### GPU During Inference
Not only training.

Inference also benefits.

Example:
```
User Prompt
     ↓
GPT Model
     ↓
Generated Response
```

GPU enables faster response generation.

### Production Relevance
Every modern AI system relies heavily on GPUs.

Examples:
| System | Uses GPU? |
|----------|-----------|
| CNNs | Yes |
| RNNs | Yes |
| LSTMs | Yes |
| Transformers | Yes |
| BERT | Yes |
| GPT | Yes |
| LLaMA | Yes |

Without GPUs:
```
Modern Deep Learning
Would Not Be Practical
```